In [ ]:
import pandas as pd


In [ ]:
imdb = pd.read_csv(
    './data/title.basics.tsv.gz',
    sep="\t",
    na_values = '\\N', 
    dtype=str,
    )

movies = imdb[imdb['titleType'] == 'movie'][
    ['tconst', 'primaryTitle']
]

In [ ]:
directors = pd.read_csv(
    './data/title.crew.tsv.gz', sep='\t', na_values='\\N', dtype=str
)[['tconst', 'directors']]

movies = movies.merge(directors, on='tconst', how='left')

In [ ]:
movies

In [ ]:
movies = movies.rename(columns={
    'tconst': 'imdb_id',
    'primaryTitle': 'title'
})

movies

In [ ]:
names = pd.read_csv(
    './data/name.basics.tsv.gz',
    sep = '\t',
    na_values = '\\N',
    dtype =str )[['nconst', 'primaryName']]

d = movies [['imdb_id', 'directors']].dropna().copy()
d['nconst'] = d['directors'].str.split(',')

d = d.explode('nconst')

d = d.merge(names, on ='nconst', how = 'left')



d['primaryName'] = d['primaryName'].fillna('')
director_names = (
    d.groupby('imdb_id')['primaryName']
    .agg(','.join)
    .rename('director')
    .reset_index()
)

movies = movies.merge(director_names, on='imdb_id', how='left')
movies = movies.drop(columns='directors')

In [ ]:
movies


In [ ]:
movies = movies.dropna(subset=['director'])
movies

In [ ]:
imdb_long = d[['imdb_id', 'nconst', 'primaryName']].rename(columns={'primaryName': 'director'})
imdb_long = imdb_long[imdb_long['director'] != '']          # cell 5 filled missing names with ''
imdb_long = imdb_long.merge(movies[['imdb_id', 'title']], on='imdb_id')   # movies already has no-director rows dropped
imdb_long = imdb_long.drop_duplicates(subset=['imdb_id', 'nconst'])
imdb_long.to_csv('data/imdb_movies.csv', index=False)